# Lab 08 — Distributed Training and Sharding Simulation
**Goal:** Simulate data parallelism and parameter sharding to understand memory and communication without requiring multiple GPUs.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
import numpy as np
n_params=1_000_000_000
ranks=8
bytes_per_param=2
print("replicated weights GiB:", n_params*bytes_per_param/1024**3)


## Prediction
Predict how DDP and sharding differ in GPU memory per rank for a 1B-parameter model.


In [ ]:
rep=n_params*bytes_per_param
sharded=rep/ranks
print("8-way shard GiB:",sharded/1024**3)
print("replicated -> sharded reduction:",rep/sharded)


## Interpretation
DDP replicates model state on each rank; sharding reduces per-rank residency but increases communication/coordination complexity.


In [ ]:
# Communication thought experiment: every rank needs gradients synchronized in classic DDP.
# FSDP-style sharding changes what is resident on each rank and introduces all-gather/reduce-scatter patterns.
tensor_bytes=512*1024**2
print("example tensor MiB:",tensor_bytes/1024**2)
print("8 ranks, ideal equal shard MiB:",tensor_bytes/8/1024**2)


## Answers
FSDP2 shards parameters, gradients and optimizer state so models that do not fit on a single accelerator can become trainable. The tradeoff is communication and implementation complexity, so benchmark the complete training step.

### Primary source
https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html
